In [9]:
# from https://techwasti.com/python-tutorial-writing-python-data-class-to-csv
def dataclass_to_csv_row(csv_writer, row):
    csv_writer.writerow(row.__dict__.values())
def dataclass_to_csv_header(csv_writer, header):
    csv_writer.writerow(header.__annotations__.keys())


In [10]:
import xml.etree.ElementTree as ET
from dataclasses import dataclass

@dataclass
class Taxon:
    taxid: int
    name: str
    rank: str

@dataclass
class LineageInfo:
    species: Taxon
    genus: Taxon
    family: Taxon
    order: Taxon
    phylum: Taxon
    kingdom: Taxon
    domain: Taxon

def parse_taxon(taxon: ET.Element):
    """Extract TaxID, rank, scientific name, and human-readable rank from a <Taxon> element."""
    taxid = int(taxon.find("TaxId").text)
    name = str(taxon.find("ScientificName").text)
    rank = str(taxon.find("Rank").text if taxon.find("Rank") is not None else "no rank")
    return Taxon(taxid, name, rank)

def parse_xml(file_path: str, csv_writer):
    """Parse XML file and extract taxonomical information."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    for taxon in root.findall("Taxon"):
        # Extract species-level info
        tax_data = parse_taxon(taxon)
        lineage = taxon.find('Lineage').text
        lin_arr = lineage.split('; ')[-7:]
        csv_writer.writerow([tax_data.taxid, tax_data.name, *lin_arr])
""" 
        # Extract lineage info
        lineage = taxon.find("LineageEx")
        if lineage is not None:
            print("\nLineage Information:")
            for ancestor in lineage.findall("Taxon")[-7:]:
                lin_data = parse_taxon(ancestor)
                print(lin_data)
        dataclass_to_csv_row(csv_writer, tax_data) """


' \n        # Extract lineage info\n        lineage = taxon.find("LineageEx")\n        if lineage is not None:\n            print("\nLineage Information:")\n            for ancestor in lineage.findall("Taxon")[-7:]:\n                lin_data = parse_taxon(ancestor)\n                print(lin_data)\n        dataclass_to_csv_row(csv_writer, tax_data) '

In [11]:
import csv

with open('data/tax.csv', 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['tax_id', 'species', 'domain', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus'])
    parse_xml('data/taxonomy_result.xml', csv_writer)


In [30]:
import pandas as pd

tax = pd.read_csv('data/tax.csv')
tax = tax.dropna(subset='genus')
tax = tax[~tax['genus'].str.startswith('unclassified')]
tax = tax[tax['domain'] == 'Eukaryota']
tax.to_csv('data/tax-filtered.csv', sep='\t', index=False)